# Planning Agent — Integration Tests (Docker)

Rulează toate testele împotriva serverului **real** pornit din Docker.  
Tool callback-urile sunt interceptate de un server HTTP local (port 9001)  
accesibil din container prin `host.docker.internal:9001`.

---
**Pre-condiții**
```bash
# din Server/
docker compose -f docker/docker-compose.dev.yml up --build -d
```
**Dependențe notebook** (o dată):
```bash
pip install httpx aiohttp python-dotenv
```

---
**Scenarii acoperite**
| # | Tip | Ce testăm |
|---|-----|-----------|
| 1 | Întrebare simplă | Planul = 1 pas LLM (fără tool-uri) |
| 2 | Sugestie / recomandare | La fel — răspuns direct, fără planning elaborat |
| 3 | 1 tool call | Plan cu 1 pas tool |
| 4 | Multi-step cu tool-uri | Listare → citire → rezumat |
| 5 | Conversație multi-turn | Continuare chat cu context |


In [36]:
# ── 0. Configurare ────────────────────────────────────────────────────────────

SERVER_URL       = 'http://localhost:8000'    # serverul din Docker

# callback_url livrat server-ului — trebuie să fie accesibil DIN container
# Windows/Mac:  host.docker.internal
# Linux:        172.17.0.1  (sau IP-ul bridge-ului Docker)
DOCKER_HOST      = 'host.docker.internal'
MOCK_PORT        = 9001
MOCK_BASE_URL    = f'http://{DOCKER_HOST}:{MOCK_PORT}'

TEST_EMAIL       = 'agent_test@example.com'
TEST_PASSWORD    = 'TestPass123!'

print(f'Server:       {SERVER_URL}')
print(f'Mock tools:   {MOCK_BASE_URL}')

Server:       http://localhost:8000
Mock tools:   http://host.docker.internal:9001


In [37]:
# ── 1. Server mock pentru tool callback-uri ───────────────────────────────────

import threading, json
from http.server import HTTPServer, BaseHTTPRequestHandler

# ── Structura virtuală a drive-ului mock ──────────────────────────────────────
MOCK_DRIVE = {
    '/':               ['raport_Q1.pdf', 'factura_001.docx', 'prezentare.pptx'],
    '/Documente':      ['contract_2024.pdf', 'nota_interna.txt'],
    '/Imagini':        ['logo.png', 'banner.jpg', 'screenshot.png'],
    '/Arhivă':         [],
    '/Arhivă/2024':    ['raport_Q4_2023.pdf'],
}

MOCK_FILE_CONTENT = {
    'raport_Q1.pdf':     'Raport financiar Q1 2024\nVânzări totale: 125.000 RON\nProfit net: 38.500 RON\nClienți noi: 47\nClienți activi: 312',
    'factura_001.docx':  'Factură nr. 001/2024\nEmitent: SRL Exemplu\nSuma: 4.200 RON + TVA\nScadență: 30.04.2024',
    'prezentare.pptx':   'Prezentare strategie 2024 — 18 slide-uri\nAutor: Echipa Marketing',
    'nota_interna.txt':  'Notă internă: Ședință luni 10:00. Agenda: buget Q2, angajări noi.',
}

def _list_files(inp):
    # Normalize: treat "", None, missing → "/" (root)
    folder = (inp.get('folder') or '/').strip()
    if not folder.startswith('/'):
        folder = '/' + folder
    folder = folder.rstrip('/') or '/'
    files = MOCK_DRIVE.get(folder, MOCK_DRIVE.get('/', []))
    return {'result': files}

def _read_file(inp):
    path     = inp.get('path', '')
    filename = path.split('/')[-1] if path else ''
    content  = MOCK_FILE_CONTENT.get(filename,
        f'[Fișierul "{path}" nu a fost găsit în mock drive]')
    return {'result': content}

MOCK_TOOL_DATA = {
    'list_files':    _list_files,
    'read_file':     _read_file,
    'move_file':     lambda inp: {'result': f'Fișierul "{inp.get("source")}" a fost mutat în "{inp.get("destination")}" cu succes.'},
    'create_folder': lambda inp: {'result': f'Directorul "{inp.get("path")}" a fost creat cu succes.'},
    'delete_file':   lambda inp: {'result': f'Fișierul "{inp.get("path")}" a fost șters cu succes.'},
    'search_files':  lambda inp: {
        'result': [f for fs in MOCK_DRIVE.values()
                   for f in fs if inp.get('query', '').lower() in f.lower()]
    },
}

_mock_calls: list[dict] = []

class ToolCallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(b'{"status":"ok"}')

    def do_POST(self):
        length   = int(self.headers.get('Content-Length', 0))
        body     = json.loads(self.rfile.read(length)) if length else {}
        tool     = body.get('tool', 'unknown')
        inp      = body.get('input', {})

        handler  = MOCK_TOOL_DATA.get(tool)
        response = handler(inp) if handler else {'result': f'Mock OK — tool={tool}'}
        _mock_calls.append({'tool': tool, 'input': inp, 'response': response})

        raw = json.dumps(response, ensure_ascii=False).encode('utf-8')
        self.send_response(200)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def log_message(self, *_):
        pass

_mock_server = HTTPServer(('0.0.0.0', MOCK_PORT), ToolCallbackHandler)
_mock_thread = threading.Thread(target=_mock_server.serve_forever, daemon=True)
_mock_thread.start()

print(f'✅ Mock tool server pornit pe :{MOCK_PORT}')
print('   Drive mock:')
for folder, files in MOCK_DRIVE.items():
    print(f'   {folder or "/"}/ → {files or "(gol)"}')
print(f'\n   Docker va apela: {MOCK_BASE_URL}/tool')

✅ Mock tool server pornit pe :9001
   Drive mock:
   // → ['raport_Q1.pdf', 'factura_001.docx', 'prezentare.pptx']
   /Documente/ → ['contract_2024.pdf', 'nota_interna.txt']
   /Imagini/ → ['logo.png', 'banner.jpg', 'screenshot.png']
   /Arhivă/ → (gol)
   /Arhivă/2024/ → ['raport_Q4_2023.pdf']

   Docker va apela: http://host.docker.internal:9001/tool


In [38]:
# ── 1b. Diagnosticare conectivitate + firewall ────────────────────────────────
# Rulează înainte de orice test.
# Pe Windows trebuie să adăugăm o regulă de firewall inbound pentru portul 9001
# altfel Docker (WSL2) nu poate ajunge la mock server-ul de pe host.

import urllib.request, subprocess, platform, socket, time, json as _json

RULE_NAME = f'NotebookMockTool_{MOCK_PORT}'

# ── 1. Mock server accesibil local? ───────────────────────────────────────────
time.sleep(0.4)
try:
    req = urllib.request.Request(f'http://localhost:{MOCK_PORT}/tool', method='GET')
    with urllib.request.urlopen(req, timeout=2): pass
    print(f'✅ Mock server rulează pe localhost:{MOCK_PORT}')
except Exception as e:
    print(f'❌ Mock server NU răspunde pe localhost:{MOCK_PORT}: {e}')
    print('   → Repornește celula 1.')

# ── 2. Firewall Windows ────────────────────────────────────────────────────────
if platform.system() == 'Windows':
    # Verifică dacă regula există deja
    check = subprocess.run(
        ['netsh', 'advfirewall', 'firewall', 'show', 'rule', f'name={RULE_NAME}'],
        capture_output=True, text=True
    )
    if 'No rules match' not in check.stdout and check.returncode == 0:
        print(f'✅ Regulă firewall deja prezentă ({RULE_NAME})')
    else:
        # Încearcă să adaugi direct (funcționează dacă procesul e admin)
        add = subprocess.run(
            ['netsh', 'advfirewall', 'firewall', 'add', 'rule',
             f'name={RULE_NAME}', 'dir=in', 'action=allow',
             'protocol=TCP', f'localport={MOCK_PORT}', 'profile=any'],
            capture_output=True, text=True
        )
        if 'Ok.' in add.stdout:
            print(f'✅ Regulă firewall adăugată pentru TCP:{MOCK_PORT}')
        else:
            # Încearcă cu PowerShell + RunAs (admin elevation)
            ps_cmd = (
                f'New-NetFirewallRule -DisplayName "{RULE_NAME}" '
                f'-Direction Inbound -Protocol TCP -LocalPort {MOCK_PORT} '
                f'-Action Allow -Profile Any'
            )
            ps = subprocess.run(
                ['powershell', '-Command', ps_cmd],
                capture_output=True, text=True
            )
            if ps.returncode == 0:
                print(f'✅ Regulă firewall adăugată via PowerShell (TCP:{MOCK_PORT})')
            else:
                print(f'⚠  Firewall: necesită drepturi de administrator.')
                print(f'   Rulează O SINGURĂ DATĂ în PowerShell ca Administrator:')
                print(f'   New-NetFirewallRule -DisplayName "{RULE_NAME}" -Direction Inbound -Protocol TCP -LocalPort {MOCK_PORT} -Action Allow -Profile Any')
else:
    print('ℹ  Linux/Mac — nu e necesară configurare firewall.')

# ── 3. Rezoluție host.docker.internal ─────────────────────────────────────────
try:
    ip = socket.gethostbyname('host.docker.internal')
    print(f'✅ host.docker.internal → {ip}')
except socket.gaierror:
    print('⚠  host.docker.internal nu se rezolvă din notebook (normal pe Linux).')
    print(f'   Setează DOCKER_HOST = "172.17.0.1" în celula de config.')

# ── 4. Test POST real la mock ──────────────────────────────────────────────────
try:
    payload = _json.dumps({'tool': 'list_files', 'input': {}}).encode()
    req = urllib.request.Request(
        f'http://localhost:{MOCK_PORT}/tool',
        data=payload, headers={'Content-Type': 'application/json'}, method='POST'
    )
    with urllib.request.urlopen(req, timeout=2) as r:
        data = _json.loads(r.read())
        print(f'✅ POST mock test OK → list_files() = {data["result"]}')
except Exception as e:
    print(f'❌ POST mock test eșuat: {e}')

# ── 5. Reamintire: repornește Docker după schimbarea docker-compose ─────────────
print()
print('⚠  Dacă ai modificat docker-compose.dev.yml (extra_hosts), repornește Docker:')
print('   docker compose -f Server/docker/docker-compose.dev.yml up --build -d')

❌ Mock server NU răspunde pe localhost:9001: timed out
   → Repornește celula 1.
⚠  Firewall: necesită drepturi de administrator.
   Rulează O SINGURĂ DATĂ în PowerShell ca Administrator:
   New-NetFirewallRule -DisplayName "NotebookMockTool_9001" -Direction Inbound -Protocol TCP -LocalPort 9001 -Action Allow -Profile Any
✅ host.docker.internal → 192.168.1.165
❌ POST mock test eșuat: timed out

⚠  Dacă ai modificat docker-compose.dev.yml (extra_hosts), repornește Docker:
   docker compose -f Server/docker/docker-compose.dev.yml up --build -d


In [39]:
# ── 1b. Diagnosticare conectivitate ──────────────────────────────────────────
# Verifică dacă mock server-ul e accesibil local și încearcă să adaugă o
# regulă de firewall Windows astfel încât Docker să poată ajunge la el.

import urllib.request, subprocess, platform, socket, time

# ── Test 1: mock server accesibil din notebook? ───────────────────────────────
time.sleep(0.3)   # lasă serverul să pornească
try:
    req = urllib.request.Request(f'http://localhost:{MOCK_PORT}/tool', method='GET')
    with urllib.request.urlopen(req, timeout=2) as r:
        print(f'✅ Mock server accesibil din notebook (localhost:{MOCK_PORT})')
except Exception as e:
    print(f'❌ Mock server NU răspunde: {e}')
    print('   → Repornește celula anterioară.')

# ── Test 2: Firewall Windows — adaugă regulă inbound pentru MOCK_PORT ─────────
if platform.system() == 'Windows':
    rule_name = f'NotebookMockTool_{MOCK_PORT}'
    r = subprocess.run(
        ['netsh', 'advfirewall', 'firewall', 'add', 'rule',
         f'name={rule_name}', 'dir=in', 'action=allow',
         'protocol=TCP', f'localport={MOCK_PORT}'],
        capture_output=True, text=True
    )
    if 'Ok.' in r.stdout:
        print(f'✅ Regulă firewall adăugată: TCP in :{MOCK_PORT}')
    elif 'already exists' in r.stdout.lower() or r.returncode == 0:
        print(f'✅ Regulă firewall deja existentă pentru :{MOCK_PORT}')
    else:
        print(f'⚠  Nu s-a putut adăuga regula firewall (necesită admin).')
        print(f'   Rulează o dată ca Administrator în CMD:')
        print(f'   netsh advfirewall firewall add rule name=MockTool dir=in action=allow protocol=TCP localport={MOCK_PORT}')
else:
    print('Linux/Mac: nu e necesară configurare firewall suplimentară.')

# ── Test 3: Rezoluție host.docker.internal ────────────────────────────────────
try:
    ip = socket.gethostbyname('host.docker.internal')
    print(f'✅ host.docker.internal → {ip}')
except socket.gaierror:
    print('⚠  host.docker.internal nu se rezolvă din notebook.')
    print('   Pe Linux setează: DOCKER_HOST = "172.17.0.1" în celula de config.')

# ── Test 4: Simulare POST de tool (verifică că mock-ul returnează date corecte) ─
try:
    payload = json.dumps({'tool': 'list_files', 'input': {'folder': '/'}}).encode()
    req = urllib.request.Request(
        f'http://localhost:{MOCK_PORT}/tool',
        data=payload,
        headers={'Content-Type': 'application/json'},
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=2) as r:
        result = json.loads(r.read())
        print(f'✅ Mock tool test OK → list_files("/") = {result["result"]}')
except Exception as e:
    print(f'❌ Test mock tool eșuat: {e}')

print('\n✔ Dacă toate verificările sunt OK, mock server-ul este gata.')

❌ Mock server NU răspunde: timed out
   → Repornește celula anterioară.
⚠  Nu s-a putut adăuga regula firewall (necesită admin).
   Rulează o dată ca Administrator în CMD:
   netsh advfirewall firewall add rule name=MockTool dir=in action=allow protocol=TCP localport=9001
✅ host.docker.internal → 192.168.1.165
❌ Test mock tool eșuat: timed out

✔ Dacă toate verificările sunt OK, mock server-ul este gata.


In [40]:
# ── 2. Definire tool-uri (cu callback_url spre mock server) ───────────────────

TOOLS = [
    {
        'name': 'list_files',
        'description': 'Listează fișierele din drive-ul virtual al utilizatorului.',
        'parameters': {'type': 'object', 'properties': {
            'folder': {'type': 'string', 'description': 'Director de listat (implicit: root)'}
        }, 'required': []},
        'callback_url': f'{MOCK_BASE_URL}/tool'
    },
    {
        'name': 'read_file',
        'description': 'Citește conținutul unui fișier text din drive.',
        'parameters': {'type': 'object', 'properties': {
            'path': {'type': 'string', 'description': 'Calea fișierului de citit'}
        }, 'required': ['path']},
        'callback_url': f'{MOCK_BASE_URL}/tool'
    },
    {
        'name': 'move_file',
        'description': 'Mută sau redenumește un fișier în drive.',
        'parameters': {'type': 'object', 'properties': {
            'source':      {'type': 'string', 'description': 'Calea sursă'},
            'destination': {'type': 'string', 'description': 'Calea destinație'}
        }, 'required': ['source', 'destination']},
        'callback_url': f'{MOCK_BASE_URL}/tool'
    },
    {
        'name': 'create_folder',
        'description': 'Creează un director nou în drive.',
        'parameters': {'type': 'object', 'properties': {
            'path': {'type': 'string', 'description': 'Calea directorului de creat'}
        }, 'required': ['path']},
        'callback_url': f'{MOCK_BASE_URL}/tool'
    },
    {
        'name': 'delete_file',
        'description': 'Șterge un fișier din drive.',
        'parameters': {'type': 'object', 'properties': {
            'path': {'type': 'string', 'description': 'Calea fișierului de șters'}
        }, 'required': ['path']},
        'callback_url': f'{MOCK_BASE_URL}/tool'
    },
]

print(f'{len(TOOLS)} tool-uri definite')

5 tool-uri definite


In [41]:
# ── 3. Auth — register sau login ──────────────────────────────────────────────

import httpx, asyncio

async def auth() -> str:
    """Returns a JWT access token. Registers first; falls back to login."""
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=15) as c:
        # Try register
        r = await c.post('/api/auth/register', json={
            'email': TEST_EMAIL, 'password': TEST_PASSWORD
        })
        if r.status_code == 201:
            print(f'✅ Utilizator nou creat: {TEST_EMAIL}')
            return r.json()['access_token']

        if r.status_code == 409:  # already exists → login
            r = await c.post('/api/auth/login', data={
                'username': TEST_EMAIL, 'password': TEST_PASSWORD
            })
            r.raise_for_status()
            print(f'✅ Login reușit: {TEST_EMAIL}')
            return r.json()['access_token']

        r.raise_for_status()

async def get_or_create_agent_key(jwt: str) -> str:
    """Returns the agent X-API-Key, creating it if needed."""
    headers = {'Authorization': f'Bearer {jwt}'}
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=15) as c:
        r = await c.post('/api/agent/register', headers=headers)
        if r.status_code == 201:
            key = r.json()['api_key']
            print(f'✅ Cheie agent nouă: {key[:8]}…')
            return key
        if r.status_code == 409:  # already exists → get it
            r = await c.get('/api/agent/key', headers=headers)
            r.raise_for_status()
            key = r.json()['api_key']
            print(f'✅ Cheie agent existentă: {key[:8]}…')
            return key
        r.raise_for_status()

JWT       = await auth()
AGENT_KEY = await get_or_create_agent_key(JWT)

AGENT_HEADERS = {'X-API-Key': AGENT_KEY}
print(f'\nGata de test — X-API-Key: {AGENT_KEY[:8]}…')

✅ Login reușit: agent_test@example.com
✅ Cheie agent existentă: 214383db…

Gata de test — X-API-Key: 214383db…


In [42]:
# ── 4. Utilitare — creare chat + stream SSE ───────────────────────────────────

async def create_chat(title: str = '', tools: list | None = None) -> str:
    """POST /api/agent/chats — returns chat_id."""
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=15) as c:
        r = await c.post('/api/agent/chats',
                         headers=AGENT_HEADERS,
                         json={'title': title, 'tools': tools or TOOLS})
        r.raise_for_status()
        return r.json()['chat_id']


async def stream_message(chat_id: str, message: str, tools: list | None = None):
    """
    POST /api/agent/chats/{chat_id}/message with SSE streaming.
    Yields parsed event dicts. Stops at [DONE].
    """
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=180) as c:
        async with c.stream(
            'POST',
            f'/api/agent/chats/{chat_id}/message',
            headers={**AGENT_HEADERS, 'Accept': 'text/event-stream'},
            json={'message': message, 'tools': tools if tools is not None else TOOLS},
        ) as resp:
            resp.raise_for_status()
            async for raw_line in resp.aiter_lines():
                if not raw_line.startswith('data: '):
                    continue
                data = raw_line[6:]
                if data.strip() == '[DONE]':
                    return
                try:
                    yield json.loads(data)
                except json.JSONDecodeError:
                    pass


print('Utilitare OK')

Utilitare OK


In [43]:
# ── 5. Renderer de evenimente SSE ─────────────────────────────────────────────

_streaming = False

def render(evt: dict):
    """Print one SSE event in a human-readable way."""
    global _streaming
    t   = evt.get('type', '?')
    msg = evt.get('message', '')

    # ── Inline streaming — no newlines between chunks ──────────────────────
    if t in ('llm_chunk', 'final_chunk'):
        if not _streaming:
            print()
            _streaming = True
        print(evt.get('content', ''), end='', flush=True)
        return

    if _streaming:
        print()  # close the last streaming line
        _streaming = False

    # ── Plan block ─────────────────────────────────────────────────────────
    if t == 'plan':
        steps = evt.get('steps', [])
        print(f'\n📋 {msg}')
        for s in steps:
            k    = s.get('type', 'llm')
            icon = '🔧' if k == 'tool' else '🧠'
            lbl  = s.get('tool', 'LLM') if k == 'tool' else 'LLM'
            print(f'   {s["id"]}. [{icon} {lbl}] {s["description"]}')
        return

    # ── Final response ──────────────────────────────────────────────────────
    if t == 'final':
        print(f'\n{"─" * 60}')
        print(f'🎯 Răspuns final:\n{evt.get("response", "")}')
        print('─' * 60)
        return

    # ── Step start — pick icon by type ─────────────────────────────────────
    if t == 'step_start':
        icon = '🔧' if evt.get('step_type') == 'tool' else '🧠'
        print(f'\n{icon} {msg}')
        return

    # ── Everything else — icon + message ───────────────────────────────────
    ICON = {
        'status':      '⏳',
        'tool_call':   '📞',
        'tool_result': '✅',
        'tool_error':  '❌',
        'llm_start':   '💭',
        'step_done':   '  ✓',
        'error':       '🚨',
    }
    icon   = ICON.get(t, '  •')
    indent = '   ' if t not in ('status', 'error') else ''
    if msg:
        print(f'{indent}{icon} {msg}')


async def run(chat_id: str, message: str, tools: list | None = None) -> dict:
    """Run one message and collect + display all events. Returns summary."""
    events, final = [], ''
    plan_steps, tool_calls = [], []

    async for evt in stream_message(chat_id, message, tools):
        events.append(evt)
        render(evt)
        if evt.get('type') == 'plan':     plan_steps  = evt.get('steps', [])
        if evt.get('type') == 'tool_call': tool_calls.append(evt.get('tool'))
        if evt.get('type') == 'final':    final = evt.get('response', '')

    return {'events': events, 'plan': plan_steps, 'tools_called': tool_calls, 'final': final}


print('Renderer OK')

Renderer OK


---
## TEST 1 — Întrebare simplă (fără tool-uri)
**Așteptat**: plan cu 1 singur pas LLM, fără niciun tool_call.

In [44]:
_mock_calls.clear()

chat1 = await create_chat(title='Test 1 — Întrebare simplă')
print(f'Chat: {chat1}\n')
print('👤 User: Ce este upscaling-ul de imagini și cum funcționează?')
print('=' * 60)

r1 = await run(chat1, 'Ce este upscaling-ul de imagini și cum funcționează?')

# ── Verificări ────────────────────────────────────────────────────────────
n_steps   = len(r1['plan'])
n_tools   = len(r1['tools_called'])
n_mock    = len(_mock_calls)
llm_only  = all(s.get('type') == 'llm' for s in r1['plan'])

print(f'\n🧪 VERIFICĂRI TEST 1')
print(f'   Pași în plan:        {n_steps}   (așteptat: 1)')
print(f'   Tool-uri apelate:    {n_tools}   (așteptat: 0)')
print(f'   HTTP calls la mock:  {n_mock}    (așteptat: 0)')
print(f'   Doar pași LLM:       {llm_only}  (așteptat: True)')

status1 = '✅ PASS' if n_steps == 1 and n_tools == 0 and llm_only else '❌ FAIL'
print(f'\n   Rezultat: {status1}')

Chat: 877bd455-87e7-48aa-837b-bbd929d0b587

👤 User: Ce este upscaling-ul de imagini și cum funcționează?
⏳ Analizez cererea și creez planul de execuție…

📋 Plan gata: 1 pas (1 LLM)
   1. [🧠 LLM] Provide an explanation of image upscaling and its mechanisms

🧠 Pasul 1/1: Provide an explanation of image upscaling and its mechanisms
   💭 Mă gândesc: Provide an explanation of image upscaling and its mechanisms…

**Upscaling‑ul de imagini – ce este și cum funcționează**

**Definiție**  
Upscaling‑ul (sau super‑rezoluția) este procesul de mărire a rezoluţiei unei imagini prin adăugarea de pixeli noi, astfel încât dimensiunea finală să fie mai mare, dar să se păstreze cât mai mult din claritatea vizuală.

---

### 1. Metode tradiționale (interpolare)
| Metodă | Principiu | Pro | Contra |
|--------|-----------|-----|--------|
| **Nearest‑neighbor** | Copiază valoarea pixelului cel mai apropiat | Rapid, simplu | Margini „blocate”, artefacte evidente |
| **Bilinear** | Interpolează liniar valoril

---
## TEST 2 — Sugestie / recomandare (fără tool-uri)
**Așteptat**: răspuns direct, fără planificare elaborată sau tool-uri.

In [45]:
_mock_calls.clear()

chat2 = await create_chat(title='Test 2 — Sugestie')
print(f'Chat: {chat2}\n')

msg2 = 'Am un drive cu sute de fișiere dezorganizate. Ce strategie de organizare îmi recomanzi?'
print(f'👤 User: {msg2}')
print('=' * 60)

r2 = await run(chat2, msg2)

n_steps2  = len(r2['plan'])
n_tools2  = len(r2['tools_called'])

print(f'\n🧪 VERIFICĂRI TEST 2')
print(f'   Pași în plan:        {n_steps2}   (așteptat: 1)')
print(f'   Tool-uri apelate:    {n_tools2}   (așteptat: 0)')

status2 = '✅ PASS' if n_steps2 == 1 and n_tools2 == 0 else '❌ FAIL'
print(f'   Rezultat: {status2}')

Chat: 4eeedd65-0c59-463b-995f-e91ff1d83886

👤 User: Am un drive cu sute de fișiere dezorganizate. Ce strategie de organizare îmi recomanzi?
⏳ Analizez cererea și creez planul de execuție…

📋 Plan gata: 1 pas (1 LLM)
   1. [🧠 LLM] Provide a concise, actionable strategy for organizing a large number of unstructured files in the user's drive.

🧠 Pasul 1/1: Provide a concise, actionable strategy for organizing a large number of unstructured files in the user's drive.
   💭 Mă gândesc: Provide a concise, actionable strategy for organizing a large number of unstructured files in the user's drive.…

**Strategie de organizare în pași clari**

1. **Definește o structură ierarhică**  
   - Categorie principală → subcategorii.  
   - Exemple:  
     - `Documente` → `Contracte`, `Rapoarte`, `Proceduri`  
     - `Imagini` → `Evenimente`, `Produse`, `Design`  
     - `Video` → `Prezentări`, `Tutoriale`  
     - `Proiecte` → `Proiect_A`, `Proiect_B`  
     - `Facturi` → `2024`, `2023` etc.

2. **Adopt

---
## TEST 3 — Un singur tool call
**Așteptat**: plan cu 1 pas tool (`list_files`) + callback HTTP la mock server.

In [46]:
_mock_calls.clear()

chat3 = await create_chat(title='Test 3 — Single tool')
print(f'Chat: {chat3}\n')

msg3 = 'Listează toate fișierele din drive-ul meu.'
print(f'👤 User: {msg3}')
print('=' * 60)

r3 = await run(chat3, msg3)

tool_names3 = r3['tools_called']
mock_calls3  = [c['tool'] for c in _mock_calls]

print(f'\n🧪 VERIFICĂRI TEST 3')
print(f'   Tool-uri planificate: {tool_names3}  (așteptat: list_files)')
print(f'   HTTP calls la mock:   {mock_calls3}  (așteptat: [list_files])')
print(f'   Input primit la mock: {[c["input"] for c in _mock_calls]}')

status3 = '✅ PASS' if 'list_files' in tool_names3 and 'list_files' in mock_calls3 else '❌ FAIL'
print(f'   Rezultat: {status3}')

Chat: b32cbd45-b3fc-43d2-b4d8-47ac6e8f2d61

👤 User: Listează toate fișierele din drive-ul meu.
⏳ Analizez cererea și creez planul de execuție…

📋 Plan gata: 1 pas (1 tool)
   1. [🔧 list_files] Listare a tuturor fișierelor din directorul rădăcină al drive-ului

🔧 Pasul 1/1: Listare a tuturor fișierelor din directorul rădăcină al drive-ului
   📞 Apelez tool: list_files(folder="")
   ✅ Eroare de la list_files: Tool 'list_files' — request timed out after 30 s (http://host.docker.int
     ✓ Pasul 1 finalizat
⏳ Am completat toți cei 1 pași. Formulez răspunsul final…

Îmi pare rău, dar nu am reușit să obțin lista fișierelor din drive‑ul tău – cererea a expirat înainte să primesc un răspuns. 

Poți încerca una dintre următoarele opțiuni:

1. **Reîncearcă comanda** – este posibil ca conexiunea să fie temporar instabilă.  
2. **Verifică permisiunile** – asigură‑te că aplicația are acces la drive‑ul tău și că nu există restricții de tip firewall sau de securitate care să blocheze solicitarea.  
3

---
## TEST 4 — Multi-step: listare → citire → rezumat
**Așteptat**: plan cu cel puțin 3 pași (tool + tool + llm) și 2+ callback-uri HTTP.

In [47]:
_mock_calls.clear()

chat4 = await create_chat(title='Test 4 — Multi-step')
print(f'Chat: {chat4}\n')

msg4 = (
    'Listează fișierele din drive-ul meu, '
    'citește raportul Q1 și spune-mi care sunt vânzările și profitul.'
)
print(f'👤 User: {msg4}')
print('=' * 60)

r4 = await run(chat4, msg4)

n_steps4  = len(r4['plan'])
n_tools4  = len(r4['tools_called'])
n_mock4   = len(_mock_calls)
mock_tools4 = [c['tool'] for c in _mock_calls]

print(f'\n🧪 VERIFICĂRI TEST 4')
print(f'   Pași în plan:        {n_steps4}   (așteptat: ≥3)')
print(f'   Tool-uri apelate:    {r4["tools_called"]}')
print(f'   HTTP calls la mock:  {mock_tools4}')
print(f'   Are list_files:      {"list_files" in mock_tools4}')
print(f'   Are read_file:       {"read_file"  in mock_tools4}')

status4 = (
    '✅ PASS'
    if n_steps4 >= 3 and 'list_files' in mock_tools4 and 'read_file' in mock_tools4
    else '❌ FAIL'
)
print(f'   Rezultat: {status4}')

Chat: 9e32566a-2187-4b1e-8589-402a51ab73c5

👤 User: Listează fișierele din drive-ul meu, citește raportul Q1 și spune-mi care sunt vânzările și profitul.
⏳ Analizez cererea și creez planul de execuție…

📋 Plan gata: 3 pași (2 tooluri, 1 LLM)
   1. [🔧 list_files] Listează toate fișierele din directorul rădăcină al drive-ului
   2. [🔧 read_file] Citește conținutul fișierului raport Q1 pentru a găsi informațiile despre vânzări și profit
   3. [🧠 LLM] Extrage și prezintă valorile de vânzări și profit din raportul citit

🔧 Pasul 1/3: Listează toate fișierele din directorul rădăcină al drive-ului
   📞 Apelez tool: list_files(folder="")
   ✅ Eroare de la list_files: Tool 'list_files' — request timed out after 30 s (http://host.docker.int
     ✓ Pasul 1 finalizat

🔧 Pasul 2/3: Citește conținutul fișierului raport Q1 pentru a găsi informațiile despre vânzări și profit
   📞 Apelez tool: read_file(path="raportul Q1")
   ✅ Eroare de la read_file: Tool 'read_file' — request timed out after 30 s (ht

---
## TEST 5 — Conversație multi-turn
Testăm că agentul ține minte contextul și nu replanifică inutil în turn-ul 2.

In [48]:
_mock_calls.clear()

chat5 = await create_chat(title='Test 5 — Multi-turn')
print(f'Chat: {chat5}\n')

# Turn 1 — cerere cu tool
msg5a = 'Listează fișierele mele.'
print(f'👤 Turn 1: {msg5a}')
print('─' * 60)
r5a = await run(chat5, msg5a)

# Turn 2 — follow-up conversațional (nu ar trebui să apeleze tool-uri)
msg5b = 'Mulțumesc! Câte fișiere are lista?'
print(f'\n👤 Turn 2: {msg5b}')
print('─' * 60)
mock_before = len(_mock_calls)
r5b = await run(chat5, msg5b)
mock_after  = len(_mock_calls)

# Turn 3 — cerere nouă cu tool
msg5c = 'Acum mută raport_Q1.pdf în /Arhivă/2024/.'
print(f'\n👤 Turn 3: {msg5c}')
print('─' * 60)
r5c = await run(chat5, msg5c)

print(f'\n🧪 VERIFICĂRI TEST 5')
print(f'   Turn 1 — tool-uri:   {r5a["tools_called"]}  (așteptat: list_files)')
print(f'   Turn 2 — tool-uri:   {r5b["tools_called"]}  (așteptat: [])')
print(f'   Turn 2 — HTTP calls: {mock_after - mock_before}  (așteptat: 0)')
print(f'   Turn 3 — tool-uri:   {r5c["tools_called"]}  (așteptat: move_file)')

ok_t1 = 'list_files' in r5a['tools_called']
ok_t2 = len(r5b['tools_called']) == 0 and (mock_after - mock_before) == 0
ok_t3 = 'move_file' in r5c['tools_called']

status5 = '✅ PASS' if ok_t1 and ok_t2 and ok_t3 else '❌ FAIL'
print(f'   Rezultat: {status5}')
if not ok_t1: print('     ⚠ Turn 1 nu a apelat list_files')
if not ok_t2: print('     ⚠ Turn 2 a apelat tool-uri inutil')
if not ok_t3: print('     ⚠ Turn 3 nu a apelat move_file')

Chat: 77e1c19d-f7d0-46d8-b4ec-c40da8f2624e

👤 Turn 1: Listează fișierele mele.
────────────────────────────────────────────────────────────
⏳ Analizez cererea și creez planul de execuție…

📋 Plan gata: 2 pași (1 tool, 1 LLM)
   1. [🔧 list_files] Listează toate fișierele din directorul rădăcină al utilizatorului
   2. [🧠 LLM] Formatează și prezintă utilizatorului lista de fișiere obținută

🔧 Pasul 1/2: Listează toate fișierele din directorul rădăcină al utilizatorului
   📞 Apelez tool: list_files(folder="")
   ✅ Eroare de la list_files: Tool 'list_files' — request timed out after 30 s (http://host.docker.int
     ✓ Pasul 1 finalizat

🧠 Pasul 2/2: Formatează și prezintă utilizatorului lista de fișiere obținută
   💭 Mă gândesc: Formatează și prezintă utilizatorului lista de fișiere obținută…

Îmi pare rău, dar lista de fișiere nu a putut fi obținută deoarece comanda anterioară a eșuat (timeout).  
Vrei să încercăm din nou să listăm fișierele din directorul rădăcină al utilizatorului?
    

---
## Sumar final

In [49]:
print('╔══════════════════════════════════════════════╗')
print('║           SUMAR TESTE PLANNING AGENT         ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Test 1 — Întrebare simplă       {status1:8s}      ║')
print(f'║  Test 2 — Sugestie               {status2:8s}      ║')
print(f'║  Test 3 — Single tool call       {status3:8s}      ║')
print(f'║  Test 4 — Multi-step             {status4:8s}      ║')
print(f'║  Test 5 — Multi-turn             {status5:8s}      ║')
print('╚══════════════════════════════════════════════╝')

print(f'\nTotal apeluri HTTP la mock server: {len(_mock_calls)}')
if _mock_calls:
    for c in _mock_calls:
        print(f'  {c["tool"]}({c["input"]})')

╔══════════════════════════════════════════════╗
║           SUMAR TESTE PLANNING AGENT         ║
╠══════════════════════════════════════════════╣
║  Test 1 — Întrebare simplă       ✅ PASS        ║
║  Test 2 — Sugestie               ✅ PASS        ║
║  Test 3 — Single tool call       ❌ FAIL        ║
║  Test 4 — Multi-step             ❌ FAIL        ║
║  Test 5 — Multi-turn             ✅ PASS        ║
╚══════════════════════════════════════════════╝

Total apeluri HTTP la mock server: 0


In [50]:
# ── Cleanup opțional: șterge chat-urile de test ───────────────────────────────
# Decomentează dacă vrei să cureți starea din memorie.

# async def cleanup():
#     async with httpx.AsyncClient(base_url=SERVER_URL, timeout=10) as c:
#         for cid in [chat1, chat2, chat3, chat4, chat5]:
#             r = await c.delete(f'/api/agent/chats/{cid}', headers=AGENT_HEADERS)
#             print(f'Deleted {cid}: {r.status_code}')
#
# await cleanup()